# PREPROCESS FOR ZALO AI LEGAL

Join `queries`, `qrels`, and `corpus`, select the most relevant chunk from each labelled document, and insert the resulting pairs into PostgreSQL.

In [1]:
from pathlib import Path

import pandas as pd

root_dir = Path.cwd().parent
data_dir = root_dir / "data" / "Legal" / "zalo-ai-legal"

queries = pd.read_parquet(data_dir / "queries.parquet")
qrels = pd.read_parquet(data_dir / "qrels.parquet")
corpus = pd.read_parquet(data_dir / "corpus.parquet")

print(f"Queries: {len(queries):,} | columns: {list(queries.columns)}")
print(f"Qrels: {len(qrels):,} | columns: {list(qrels.columns)}")
print(f"Corpus: {len(corpus):,} | columns: {list(corpus.columns)}")

Queries: 3,196 | columns: ['query_id', 'question']
Qrels: 2,556 | columns: ['corpus_id', 'query_id', 'score']
Corpus: 61,425 | columns: ['id', 'title', 'text']


In [2]:
def normalize_columns(frame, aliases, frame_name):
    columns = {}
    for canonical_name, candidates in aliases.items():
        source_name = next((name for name in candidates if name in frame.columns), None)
        if source_name is None:
            raise ValueError(
                f"{frame_name} is missing {canonical_name!r}. "
                f"Available columns: {list(frame.columns)}"
            )
        columns[source_name] = canonical_name
    return frame.rename(columns=columns)


queries = normalize_columns(
    queries,
    {"query_id": ("query_id", "query-id", "id"), "query": ("question", "query", "text")},
    "queries.parquet",
)
qrels = normalize_columns(
    qrels,
    {"query_id": ("query_id", "query-id"), "corpus_id": ("corpus_id", "corpus-id"), "score": ("score",)},
    "qrels.parquet",
)
corpus = normalize_columns(
    corpus,
    {"corpus_id": ("corpus_id", "corpus-id", "id"), "title": ("title",), "document": ("document", "text")},
    "corpus.parquet",
)

query_conflicts = queries.groupby("query_id")["query"].nunique().gt(1)
if query_conflicts.any():
    raise ValueError("Some duplicated query IDs have conflicting text.")

queries = queries.drop_duplicates(subset="query_id")
positive_qrels = qrels.loc[qrels["score"] > 0].drop_duplicates(
    subset=["query_id", "corpus_id"]
)

pairs = positive_qrels.merge(
    queries,
    on="query_id",
    how="left",
    validate="many_to_one",
).merge(
    corpus,
    on="corpus_id",
    how="left",
    validate="many_to_one",
)

if pairs[["query", "title", "document"]].isna().any().any():
    raise ValueError("Some qrels cannot be resolved to a query or corpus document.")

print(f"Unique queries: {len(queries):,}")
print(f"Resolved positive pairs: {len(pairs):,}")

Unique queries: 3,196
Resolved positive pairs: 2,556


In [3]:
import re

CHUNK_SIZE = 1_200
CHUNK_OVERLAP = 150


def chunk_content(content: str) -> list[tuple[int, str]]:
    """Split a document into overlapping character chunks."""
    chunks = []
    start = 0
    step = CHUNK_SIZE - CHUNK_OVERLAP

    while start < len(content):
        end = min(start + CHUNK_SIZE, len(content))
        chunks.append((start, content[start:end].strip()))
        if end == len(content):
            break
        start += step

    return chunks


def tokenize(text: str) -> set[str]:
    return set(re.findall(r"\w+", text.lower(), flags=re.UNICODE))


def select_positive_chunk(query: str, title: str, document: str) -> tuple[int, str]:
    """Select the chunk with the highest lexical overlap with the query."""
    query_tokens = tokenize(query)
    chunks = chunk_content(document)

    def score(chunk: tuple[int, str]) -> tuple[float, int]:
        start, text = chunk
        chunk_tokens = tokenize(f"{title} {text}")
        overlap = len(query_tokens.intersection(chunk_tokens))
        normalization = max(len(query_tokens.union(chunk_tokens)), 1)
        return overlap / normalization, -start

    return max(chunks, key=score)

In [4]:
import hashlib

id_prefix = "legal_001"
source = "Zalo AI Legal"
records = []

for row in pairs.itertuples(index=False):
    chunk_start, chunk_text = select_positive_chunk(
        query=row.query,
        title=row.title,
        document=row.document,
    )
    pair_key = f"{row.query_id}|{row.corpus_id}|{chunk_start}"
    pair_hash = hashlib.sha1(pair_key.encode("utf-8")).hexdigest()[:24]
    records.append(
        {
            "data_id": f"{id_prefix}_{pair_hash}",
            "source": source,
            "title": row.title,
            "anchor": row.query,
            "positive": f"{row.title}\n{chunk_text}",
            "hard_negative": None,
        }
    )

print(f"Prepared records: {len(records):,}")
print(records[0])

Prepared records: 2,556
{'data_id': 'legal_001_1c9a8e7a030caabef47194f1', 'source': 'Zalo AI Legal', 'title': 'Điều 8. Kiểm nghiệm trước khi lưu hành đối với thuốc được quy định tại Khoản 4 Điều 103 của Luật dược', 'anchor': 'Trách nhiệm của cơ sở sản xuất nhập khẩu thuốc về việc kiểm nghiệm thuốc trước khi lưu hành được quy định như thế nào?', 'positive': 'Điều 8. Kiểm nghiệm trước khi lưu hành đối với thuốc được quy định tại Khoản 4 Điều 103 của Luật dược\nc chưa nghiên cứu thiết lập được;\nc) Chỉ được đưa ra lưu thông, phân phối các lô thuốc đã có kết quả kiểm nghiệm đạt tiêu chuẩn chất lượng.\n7. Việc kiểm nghiệm vắc xin, sinh phẩm là huyết thanh chứa kháng thể, dẫn xuất của máu và huyết tương người được thực hiện theo quy định tại Điều 10 và Điều 11 Thông tư này.', 'hard_negative': None}


In [5]:
from sqlalchemy import select

from database.models import LegalModel
from database.sql_manager import SQL_Manager

sql_mng = SQL_Manager()
sql_mng.create_legal_model()
existing_ids = set(
    sql_mng.con.scalars(
        select(LegalModel.data_id).where(LegalModel.data_id.like(f"{id_prefix}_%"))
    )
)

inserted = 0
skipped = 0

try:
    for record in records:
        if record["data_id"] in existing_ids:
            skipped += 1
            continue

        sql_mng.insert_legal_model(LegalModel(**record))
        inserted += 1

        if inserted % 1_000 == 0:
            sql_mng.con.commit()

    sql_mng.con.commit()
finally:
    sql_mng.close()

print(f"Inserted: {inserted:,}; skipped existing: {skipped:,}")

Inserted: 2,556; skipped existing: 0
